# HyperDream Colab 分步实验（大白话版）

这份 Notebook 的目标：让你**看得到每一步**，并且每一步都知道在干什么。

```text
实验流程（Flow）
├─ 1) 挂载 Google Drive（可选）
├─ 2) 拉取项目代码（clone / pull）
├─ 3) 安装依赖
├─ 4) 设置实验参数（run_id、epoch 等）
├─ 5) 跑测试（确保代码没坏）
├─ 6) 跑 baseline / transfer / ablation / robustness
├─ 7) 生成图表
└─ 8) 同步结果到 Drive + 下次 resume 续训
```

你可以先跑一个小规模 smoke（快速验证），再扩大训练预算。

## Cell 1：挂载 Google Drive（可选）

大白话：
- 如果你担心 Colab 断线丢结果，就先挂载 Drive。
- 挂载后，我们可以把 `results/`、`checkpoints/` 同步过去。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## Cell 2：拉取项目代码（首次 clone，之后自动 pull）

大白话：
- 第一次运行：下载仓库。
- 以后再运行：自动拉最新代码。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/peter941221/High_Dimensional_WorldModel.git'
PROJECT_DIR = Path('/content/High_Dimensional_WorldModel')
BRANCH = 'main'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    try:
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    except subprocess.CalledProcessError:
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)

    pull = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH],
        text=True,
        capture_output=True,
    )
    if pull.returncode != 0:
        if pull.stdout:
            print(pull.stdout)
        if pull.stderr:
            print(pull.stderr)
        print('⚠️ 检测到本地分支无法 ff-only，自动对齐到 origin/main（Colab 防卡死）')
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.chdir(PROJECT_DIR)
print('当前目录:', Path.cwd())

## Cell 3：安装依赖

大白话：
- 就是把运行项目需要的 Python 包装好。
- 这一步只要不报错，后面才能跑。

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## Cell 4：设置实验参数（最关键）

大白话：
- `RUN_ID`：实验名字，后续用它续训。
- `RESUME`：是否从上次继续。
- `*_EPOCHS`：训练轮数，先小后大。

建议：第一次把轮数设小一点，先跑通。

In [ ]:
from datetime import datetime

# 你可以手动改成固定名字，例如 'paper_v1'
RUN_ID = f'colab_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RESUME = False  # 下次续训改成 True，并保持同一个 RUN_ID

# 快速验证参数（先跑通）
BASELINE_EPOCHS = 3
TRANSFER_PRETRAIN_EPOCHS = 2
TRANSFER_FINETUNE_EPOCHS = 2
ABLATION_EPOCHS = 2
ROBUSTNESS_EPISODES = 20
EVAL_EPISODES = 10
MAX_STEPS = 80

# checkpoint 管理
SAVE_EVERY = 2   # 每2轮存一份归档
KEEP_LAST = 3    # 最多保留3份归档

RUN_TESTS = True
SYNC_TO_DRIVE = True
DRIVE_SYNC_DIR = '/content/drive/MyDrive/High_Dimensional_WorldModel_runs'

print('RUN_ID =', RUN_ID)
print('RESUME =', RESUME)

## Cell 5：先跑测试（推荐）

大白话：
- 先确认代码健康，再花时间跑实验。
- 如果这里报错，先修再继续。

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
else:
    print('跳过测试（RUN_TESTS=False）')

## Cell 6：按顺序跑四个实验 + 可视化

大白话：
- baseline：各维度单独训练的基线。
- transfer：高维预训练再迁移到3D。
- ablation：对比不同世界模型结构。
- robustness：干扰条件下鲁棒性。
- visualize：画图。

In [ ]:
import subprocess

common = [
    '--run-id', RUN_ID,
    '--max-steps', str(MAX_STEPS),
    '--eval-episodes', str(EVAL_EPISODES),
    '--save-every', str(SAVE_EVERY),
    '--keep-last', str(KEEP_LAST),
]
if RESUME:
    common.append('--resume')

# 1) baseline
subprocess.run([
    'python', 'experiments/run_baseline.py',
    *common,
    '--epochs', str(BASELINE_EPOCHS),
], check=True)

# 2) transfer
subprocess.run([
    'python', 'experiments/run_transfer.py',
    *common,
    '--pretrain-epochs', str(TRANSFER_PRETRAIN_EPOCHS),
    '--finetune-epochs', str(TRANSFER_FINETUNE_EPOCHS),
], check=True)

# 3) ablation
subprocess.run([
    'python', 'experiments/run_ablation.py',
    *common,
    '--epochs', str(ABLATION_EPOCHS),
], check=True)

# 4) robustness
robust_cmd = [
    'python', 'experiments/run_robustness.py',
    '--run-id', RUN_ID,
    '--episodes', str(ROBUSTNESS_EPISODES),
]
if RESUME:
    robust_cmd.append('--resume')
subprocess.run(robust_cmd, check=True)

# 5) 画图
subprocess.run(['python', 'experiments/visualize.py'], check=True)

print('✅ 全部实验执行完成')

## Cell 7：把结果同步到 Drive（建议）

大白话：
- 这一步是“防丢档”。
- 就算 Colab 重置，你的结果也在 Drive 里。

In [ ]:
from pathlib import Path
import shutil

if SYNC_TO_DRIVE:
    dst_root = Path(DRIVE_SYNC_DIR) / RUN_ID
    dst_root.mkdir(parents=True, exist_ok=True)
    for name in ['results', 'checkpoints', 'figures', 'report']:
        src = Path(name)
        dst = dst_root / name
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print('✅ 已同步到 Drive:', dst_root)
else:
    print('跳过 Drive 同步（SYNC_TO_DRIVE=False）')

## Cell 8：快速查看结果文件

大白话：
- 看看 json 和 png 有没有出来。
- 这是最直观的“跑完没跑完”检查。

In [ ]:
!ls -R results | head -n 200
!ls -R figures | head -n 200

## Cell 9：下一次怎么续训？

大白话：
1. 把 `RUN_ID` 改成上次那个。
2. 把 `RESUME=True`。
3. 把 epoch 调大（比如从 3 改到 10）。

这样就会从 checkpoint 接着跑，不会重头开始。

## Cell 10：可选一键版（如果你不想分步）

这个就是调用我们已经写好的自动脚本 `colab_autorun.py`。
你现在这份 Notebook 是“可视化分步版”，下面这格是“偷懒一键版”。

In [ ]:
!python colab_autorun.py \
  --mount-drive \
  --sync-to-drive \
  --run-tests \
  --run-id colab_quick_demo \
  --baseline-epochs 3 \
  --transfer-pretrain-epochs 2 \
  --transfer-finetune-epochs 2 \
  --ablation-epochs 2